# How To Use PyNarrative To Tell The Painful Truth About Safe Water

*A step-by-step Python tutorial using WHO/UNICEF safe water data to show how PyNarrative's add_context method turns a plain chart into a data narrative*

**Data:** UN SDG Global Database / WHO-UNICEF JMP, https://unstats.un.org/sdgapi/v1/sdg/Series/Data?seriesCode=SH_H2O_SAFE, data to 2024

**Article:** (link added when the article is published)

Run the cells in order. Dataset files are in the same folder as this notebook.

### The Dataset

The UN SDG Global Database tracks indicator SH_H2O_SAFE — the percentage of people with safely managed drinking water services. Data comes from the [WHO/UNICEF Joint Monitoring Programme (JMP)](https://washdata.org/), and the API is public:

In [1]:
import pandas as pd
from pynarrative import Story

df = pd.read_csv('safe-water.csv')
print(df.columns.tolist())
print(df.head())

# Filter to global ALLAREA trend
world = df[(df['location'] == 'ALLAREA') & (df['area'] == 'World')].copy()
world = world.sort_values('year')

['area_code', 'area', 'year', 'value', 'location', 'reporting_type', 'series', 'source']
   area_code         area  year      value location reporting_type  \
0          4  Afghanistan  2000   8.535748    RURAL              G   
1          4  Afghanistan  2000  11.483052  ALLAREA              G   
2          4  Afghanistan  2000  21.885258    URBAN              G   
3          4  Afghanistan  2001  21.885258    URBAN              G   
4          4  Afghanistan  2001  11.495201  ALLAREA              G   

        series                                             source  
0  SH_H2O_SAFE  WHO/UNICEF Joint Monitoring Programme for Wate...  
1  SH_H2O_SAFE  WHO/UNICEF Joint Monitoring Programme for Wate...  
2  SH_H2O_SAFE  WHO/UNICEF Joint Monitoring Programme for Wate...  
3  SH_H2O_SAFE  WHO/UNICEF Joint Monitoring Programme for Wate...  
4  SH_H2O_SAFE  WHO/UNICEF Joint Monitoring Programme for Wate...  


### Chart 1: The Plain Chart

Let's start by building a basic Story line chart of the global trend, 2000–2024, with no context at all.

In [2]:
# Chart 1 — plain chart, no context
story = Story(world, width=700, height=380)
story.mark_line()
story.encode(x='year:Q', y='value:Q')
story.add_title('Global safely managed drinking water access, 2000–2024')
chart = story.render()
chart

alt.LayerChart(...)

### Chart 2: Adding Context

Now add one `add_context()` call that explains the JMP definition in plain language.

In [3]:
# Chart 2 — one add_context, definition of "safely managed"
story = Story(world, width=700, height=380)
story.mark_line()
story.encode(x='year:Q', y='value:Q')
story.add_title('Global safely managed drinking water access, 2000–2024')

story.add_context(
    'Safely managed drinking water: an improved source that is on-premises, '
    'available when needed, and free of contamination (WHO/UNICEF JMP definition).'
)

chart = story.render()
chart

alt.LayerChart(...)

### Chart 3: The Countries The Average Hides

Charts 1 and 2 show the global trend. Chart 3 changes the question: not how has the average moved, but who is furthest behind.

In [4]:
# Chart 3 — bottom 15 countries bar chart
AGGREGATE_CODES = {1, 5, 9, 11, 13, 14, 15, 17, 18, 21, 29, 30, 34, 35, 39,
                   53, 54, 57, 61, 62, 142, 143, 145, 150, 151, 154, 155,
                   199, 202, 419, 432, 513, 722, 747, 753}

countries_2024 = df[
    (df['location'] == 'ALLAREA') &
    (df['year'] == 2024) &
    (~df['area_code'].isin(AGGREGATE_CODES))
].dropna(subset=['value'])

bottom15 = countries_2024.nsmallest(15, 'value').sort_values('value', ascending=False)

story = Story(bottom15, width=700, height=420)
story.mark_bar(orient='horizontal')
story.encode(y='area:N', x='value:Q')
story.add_title('The 15 countries furthest from safe water, 2024')

story.add_annotation(
    y_point='Central African Republic',
    x_point=6.2,
    annotation_text='6.2%',
    arrow_direction='right',
    arrow_color='#D62728',
    label_color='#D62728',
    label_size=14,
)

story.add_context(
    'All 15 countries are affected by active conflict or severe institutional fragility. '
    'The global average of 73.7% is not a floor — these countries pull it down from well below.'
)

story.add_source('UN SDG Global Database / WHO-UNICEF JMP, 2025')
chart = story.render()
chart

alt.LayerChart(...)